In [2]:
# ============================================================
# NRL ANALYTICS WEEKLY — CELL 1: SETUP (run once per session)
# ============================================================

!pip install requests beautifulsoup4 pandas -q

import requests
import pandas as pd
from bs4 import BeautifulSoup

ODDS_API_KEY = "b452724dbd9434d9874cc32fc74f2d16"  # ← paste your actual key here

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

all_teams = ['Broncos','Bulldogs','Cowboys','Dolphins','Dragons','Eels',
             'Knights','Panthers','Rabbitohs','Raiders','Roosters',
             'Sea Eagles','Sharks','Storm','Titans','Warriors','Wests Tigers']

severity_order = {
    "season": 0, "indefinite": 1, "tbc": 2,
    "6-8 weeks": 3, "6 weeks": 3, "8 weeks": 3,
    "4-6 weeks": 4, "4 weeks": 4, "5 weeks": 4,
    "2-4 weeks": 5, "3 weeks": 5, "2 weeks": 6,
    "1-2 weeks": 7, "1 week": 8, "round": 9,
}

def severity_score(return_str):
    r = str(return_str).lower().strip()
    for key, score in severity_order.items():
        if key in r:
            return score
    return 5

def get_winner(row):
    if row['HomeTeamScore'] > row['AwayTeamScore']:   return row['HomeTeam']
    elif row['AwayTeamScore'] > row['HomeTeamScore']: return row['AwayTeam']
    return 'Draw'

def get_bye_team(round_df):
    playing = set(round_df['HomeTeam'].tolist() + round_df['AwayTeam'].tolist())
    return [t for t in all_teams if t not in playing]

def form_badges(form_str):
    badges = []
    for ch in str(form_str):
        if ch == "W":
            badges.append('<span style="display:inline-block;background:#16a34a;color:#fff;padding:3px 7px;border-radius:999px;font-size:12px;font-weight:bold;margin-right:4px;">W</span>')
        elif ch == "L":
            badges.append('<span style="display:inline-block;background:#dc2626;color:#fff;padding:3px 7px;border-radius:999px;font-size:12px;font-weight:bold;margin-right:4px;">L</span>')
        elif ch == "D":
            badges.append('<span style="display:inline-block;background:#f59e0b;color:#fff;padding:3px 7px;border-radius:999px;font-size:12px;font-weight:bold;margin-right:4px;">D</span>')
    return "".join(badges)

def severity_icon(return_str):
    score = severity_score(str(return_str))
    if score <= 2:   return '<span style="font-weight:bold;">🔴 Long-term</span>'
    elif score <= 4: return '<span style="font-weight:bold;">🟠 Medium</span>'
    elif score <= 6: return '<span style="font-weight:bold;">🟡 Short</span>'
    else:            return '<span style="font-weight:bold;">🟢 Near return</span>'

def safe_odds(val):
    try:    return f"{float(val):.2f}"
    except: return "N/A"

def build_newsletter_html(round_num, results_df, ladder_df, odds_df, injuries_df, bye_teams):
    value = odds_df[
        ((odds_df['Home Odds'] >= 2.0) & (odds_df['Home Odds'] <= 3.5)) |
        ((odds_df['Away Odds'] >= 2.0) & (odds_df['Away Odds'] <= 3.5))
    ].copy()

    _inj = injuries_df.copy()
    if not _inj.empty:
        _inj['SeverityScore'] = _inj['Return'].apply(severity_score)
        _inj = _inj.sort_values(['Team','SeverityScore','Player'], ascending=[True,True,True]).reset_index(drop=True)

    html = f"""
<div style="max-width:900px;margin:0 auto;font-family:Arial,Helvetica,sans-serif;color:#111827;line-height:1.6;">
    <div style="background:#0f172a;color:#ffffff;padding:28px 24px;border-radius:16px;text-align:center;margin-bottom:24px;">
        <div style="font-size:14px;font-weight:bold;letter-spacing:1px;text-transform:uppercase;opacity:0.85;">Weekly Rugby League Brief</div>
        <h1 style="margin:10px 0 8px 0;font-size:34px;line-height:1.2;">🏉 NRL Analytics Weekly</h1>
        <div style="font-size:20px;font-weight:600;">Round {round_num} Preview</div>
    </div>
    <div style="background:#eff6ff;border:1px solid #bfdbfe;padding:14px 18px;border-radius:12px;margin-bottom:24px;">
        <span style="font-size:15px;"><strong>😴 Bye this round:</strong> {', '.join(bye_teams) if bye_teams else 'None'}</span>
    </div>

    <div style="background:#fff;border:1px solid #e5e7eb;border-radius:16px;padding:22px;margin-bottom:24px;">
        <h2 style="font-size:26px;margin:0 0 16px 0;">📋 Round {round_num - 1} Results</h2>
        <table style="width:100%;border-collapse:collapse;font-size:15px;">
            <thead><tr style="background:#f3f4f6;">
                <th style="padding:12px;border-bottom:1px solid #d1d5db;text-align:left;">Home</th>
                <th style="padding:12px;border-bottom:1px solid #d1d5db;text-align:center;">Score</th>
                <th style="padding:12px;border-bottom:1px solid #d1d5db;text-align:left;">Away</th>
                <th style="padding:12px;border-bottom:1px solid #d1d5db;text-align:left;">Winner</th>
            </tr></thead><tbody>"""

    for _, row in results_df.iterrows():
        winner_badge = f'<span style="background:#dcfce7;color:#166534;padding:5px 10px;border-radius:999px;font-size:12px;font-weight:bold;">{row["Winner"]}</span>'
        html += f"""<tr>
            <td style="padding:12px;border-bottom:1px solid #e5e7eb;">{row['HomeTeam']}</td>
            <td style="padding:12px;border-bottom:1px solid #e5e7eb;text-align:center;font-weight:bold;">{row['Score']}</td>
            <td style="padding:12px;border-bottom:1px solid #e5e7eb;">{row['AwayTeam']}</td>
            <td style="padding:12px;border-bottom:1px solid #e5e7eb;">{winner_badge}</td>
        </tr>"""

    html += f"""</tbody></table></div>

    <div style="background:#fff;border:1px solid #e5e7eb;border-radius:16px;padding:22px;margin-bottom:24px;">
        <h2 style="font-size:26px;margin:0 0 16px 0;">🏆 Ladder — Top 8</h2>
        <div style="font-size:14px;color:#6b7280;margin-bottom:14px;">After Round {round_num - 1}</div>
        <table style="width:100%;border-collapse:collapse;font-size:15px;">
            <thead><tr style="background:#f3f4f6;">
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:center;">Pos</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:left;">Team</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:center;">P</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:center;">W</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:center;">L</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:center;">Pts</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:center;">Diff</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:left;">Form</th>
            </tr></thead><tbody>"""

    for pos, (_, row) in enumerate(ladder_df.head(8).iterrows(), start=1):
        html += f"""<tr>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;text-align:center;font-weight:bold;">{pos}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;font-weight:600;">{row['Team']}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;text-align:center;">{int(row['Played'])}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;text-align:center;">{int(row['Wins'])}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;text-align:center;">{int(row['Losses'])}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;text-align:center;font-weight:bold;">{int(row['Points'])}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;text-align:center;">{int(row['Diff'])}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;">{form_badges(row['Form'])}</td>
        </tr>"""

    html += f"""</tbody></table></div>

    <div style="background:#fff;border:1px solid #e5e7eb;border-radius:16px;padding:22px;margin-bottom:24px;">
        <h2 style="font-size:26px;margin:0 0 16px 0;">🎲 Round {round_num} Odds</h2>
        <table style="width:100%;border-collapse:collapse;font-size:15px;">
            <thead><tr style="background:#f3f4f6;">
                <th style="padding:12px;border-bottom:1px solid #d1d5db;text-align:left;">Match</th>
                <th style="padding:12px;border-bottom:1px solid #d1d5db;text-align:center;">Home Odds</th>
                <th style="padding:12px;border-bottom:1px solid #d1d5db;text-align:center;">Away Odds</th>
                <th style="padding:12px;border-bottom:1px solid #d1d5db;text-align:left;">Favourite</th>
            </tr></thead><tbody>"""

    for _, row in odds_df.iterrows():
        fav_badge = f'<span style="background:#fef3c7;color:#92400e;padding:5px 10px;border-radius:999px;font-size:12px;font-weight:bold;">⭐ {row["Favourite"]}</span>'
        html += f"""<tr>
            <td style="padding:12px;border-bottom:1px solid #e5e7eb;font-weight:600;">{row['Home']} vs {row['Away']}</td>
            <td style="padding:12px;border-bottom:1px solid #e5e7eb;text-align:center;">${safe_odds(row['Home Odds'])}</td>
            <td style="padding:12px;border-bottom:1px solid #e5e7eb;text-align:center;">${safe_odds(row['Away Odds'])}</td>
            <td style="padding:12px;border-bottom:1px solid #e5e7eb;">{fav_badge}</td>
        </tr>"""

    html += """</tbody></table></div>

    <div style="background:#fff;border:1px solid #e5e7eb;border-radius:16px;padding:22px;margin-bottom:24px;">
        <h2 style="font-size:26px;margin:0 0 16px 0;">💰 Value Bets This Round</h2>"""

    if value.empty:
        html += '<div style="background:#f9fafb;padding:14px 16px;border-radius:12px;color:#6b7280;">No clear value spots this round.</div>'
    else:
        for _, row in value.iterrows():
            underdog = row['Home'] if row['Home Odds'] > row['Away Odds'] else row['Away']
            dog_odds = row['Home Odds'] if row['Home Odds'] > row['Away Odds'] else row['Away Odds']
            html += f"""<div style="background:#f9fafb;border:1px solid #e5e7eb;padding:14px 16px;border-radius:12px;margin-bottom:10px;">
                <div style="font-size:17px;font-weight:700;">👀 {underdog}</div>
                <div style="font-size:15px;color:#374151;">Worth watching at <strong>${safe_odds(dog_odds)}</strong></div>
            </div>"""

    html += """</div>

    <div style="background:#fff;border:1px solid #e5e7eb;border-radius:16px;padding:22px;margin-bottom:24px;">
        <h2 style="font-size:26px;margin:0 0 10px 0;">🏥 Injury Report</h2>
        <div style="font-size:13px;color:#6b7280;margin-bottom:16px;">
            🔴 Long-term/TBC &nbsp;&nbsp; 🟠 4–8 weeks &nbsp;&nbsp; 🟡 2–4 weeks &nbsp;&nbsp; 🟢 Near return
        </div>
        <table style="width:100%;border-collapse:collapse;font-size:14px;">
            <thead><tr style="background:#f3f4f6;">
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:left;">Player</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:left;">Team</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:left;">Injury</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:left;">Return</th>
                <th style="padding:10px;border-bottom:1px solid #d1d5db;text-align:left;">Severity</th>
            </tr></thead><tbody>"""

    for _, row in _inj.iterrows():
        html += f"""<tr>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;font-weight:600;">{row['Player']}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;">{row['Team']}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;">{row['Injury']}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;">{row['Return']}</td>
            <td style="padding:10px;border-bottom:1px solid #e5e7eb;">{severity_icon(row['Return'])}</td>
        </tr>"""

    html += """</tbody></table></div>

    <div style="text-align:center;color:#6b7280;font-size:12px;padding:10px 0 20px 0;">
        <div>📊 Data: fixturedownload.com + The Odds API + ZeroTackle</div>
        <div>💰 Bet responsibly | 18+ only | gamblinghelponline.org.au</div>
    </div>
</div>"""

    return html

print("✅ Cell 1 ready — all functions loaded")

✅ Cell 1 ready — all functions loaded


In [ ]:
# ============================================================
# NRL ANALYTICS WEEKLY — CELL 2: GENERATE (run every Thursday)
# ============================================================

# ── 1. Match data ─────────────────────────────────────────────
df = pd.DataFrame(requests.get("https://fixturedownload.com/feed/json/nrl-2026", headers=headers).json())
completed = df[df['HomeTeamScore'].notna()].copy()
latest_round = completed['RoundNumber'].max()
next_round = int(latest_round) + 1
print(f"✅ Match data loaded — latest completed round: {int(latest_round)}")

# ── 2. Last round results ──────────────────────────────────────
this_round = completed[completed['RoundNumber'] == latest_round].copy()
this_round['Winner'] = this_round.apply(get_winner, axis=1)
this_round['Score']  = this_round['HomeTeamScore'].astype(int).astype(str) + ' - ' + this_round['AwayTeamScore'].astype(int).astype(str)
bye_teams = get_bye_team(this_round)

# ── 3. Ladder ─────────────────────────────────────────────────
all_rounds = completed['RoundNumber'].unique()
bye_counts = {team: 0 for team in all_teams}
for rnd in all_rounds:
    for team in get_bye_team(completed[completed['RoundNumber'] == rnd]):
        bye_counts[team] += 1

home_games = completed.copy()
home_games['Team'] = home_games['HomeTeam']
home_games['Win']  = (home_games['HomeTeamScore'] > home_games['AwayTeamScore']).astype(int)
away_games = completed.copy()
away_games['Team'] = away_games['AwayTeam']
away_games['Win']  = (away_games['AwayTeamScore'] > away_games['HomeTeamScore']).astype(int)
home_record = home_games.groupby('Team')['Win'].agg(['sum','count']).rename(columns={'sum':'HW','count':'HP'})
away_record = away_games.groupby('Team')['Win'].agg(['sum','count']).rename(columns={'sum':'AW','count':'AP'})

home_g = completed[['HomeTeam','AwayTeam','HomeTeamScore','AwayTeamScore']].copy()
home_g.columns = ['Team','Opponent','For','Against']
away_g = completed[['AwayTeam','HomeTeam','AwayTeamScore','HomeTeamScore']].copy()
away_g.columns = ['Team','Opponent','For','Against']
all_games = pd.concat([home_g, away_g], ignore_index=True)
all_games['Win']  = (all_games['For'] > all_games['Against']).astype(int)
all_games['Loss'] = (all_games['For'] < all_games['Against']).astype(int)
all_games['Draw'] = (all_games['For'] == all_games['Against']).astype(int)
all_games['Pts']  = all_games['Win'] * 2 + all_games['Draw']
all_games['Diff'] = all_games['For'] - all_games['Against']

def get_form(team, n=5):
    games = all_games[all_games['Team'] == team].tail(n)
    return ' '.join(['W' if r==1 else ('D' if d==1 else 'L') for r,d in zip(games['Win'],games['Draw'])])

ladder = all_games.groupby('Team').agg(
    Played=('Win','count'), Wins=('Win','sum'), Losses=('Loss','sum'),
    Draws=('Draw','sum'), For=('For','sum'), Against=('Against','sum'),
    Diff=('Diff','sum'), Points=('Pts','sum')
).reset_index()
ladder['Byes']  = ladder['Team'].map(bye_counts)
ladder['Home']  = ladder['Team'].map(lambda t: f"{home_record.loc[t,'HW']}/{home_record.loc[t,'HP']}" if t in home_record.index else '0/0')
ladder['Away']  = ladder['Team'].map(lambda t: f"{away_record.loc[t,'AW']}/{away_record.loc[t,'AP']}" if t in away_record.index else '0/0')
ladder['Form']  = ladder['Team'].map(get_form)
ladder = ladder.sort_values(['Points','Diff'], ascending=False).reset_index(drop=True)
ladder.index += 1
print(f"✅ Ladder built — {len(ladder)} teams")

# ── 4. Odds ───────────────────────────────────────────────────
odds_response = requests.get(
    "https://api.the-odds-api.com/v4/sports/rugbyleague_nrl/odds",
    params={"apiKey": ODDS_API_KEY, "regions": "au", "markets": "h2h",
            "oddsFormat": "decimal", "bookmakers": "sportsbet"}
)
odds_data = odds_response.json()
odds_rows = []
for match in odds_data:
    home, away = match['home_team'], match['away_team']
    home_odds = away_odds = None
    for bm in match['bookmakers']:
        if bm['key'] == 'sportsbet':
            for o in bm['markets'][0]['outcomes']:
                if o['name'] == home:   home_odds = o['price']
                elif o['name'] == away: away_odds = o['price']
    favourite = home if (home_odds and away_odds and home_odds < away_odds) else away
    odds_rows.append({'Date': match['commence_time'][:10], 'Home': home, 'Away': away,
                      'Home Odds': home_odds, 'Away Odds': away_odds, 'Favourite': favourite})
odds_df = pd.DataFrame(odds_rows)
print(f"✅ Odds loaded — {len(odds_df)} matches | Requests remaining: {odds_response.headers.get('x-requests-remaining')}")

# ── 5. Injuries ───────────────────────────────────────────────
inj_response = requests.get("https://www.zerotackle.com/nrl/injuries-suspensions/", headers=headers)
soup = BeautifulSoup(inj_response.text, "html.parser")
tables = soup.find_all("table")
injuries = []
for table in tables:
    team_name = "Unknown"
    for sibling in table.previous_siblings:
        if hasattr(sibling, 'get_text'):
            text = sibling.get_text(strip=True)
            if text:
                team_name = text
                break
    for row in table.find_all("tr", class_="table-row-border"):
        cells = row.find_all("td")
        if len(cells) >= 3:
            injuries.append({
                "Team":   team_name,
                "Player": cells[1].get_text(separator=" ", strip=True),
                "Injury": cells[2].get_text(strip=True),
                "Return": cells[3].get_text(strip=True) if len(cells) > 3 else "TBC"
            })
injuries_df = pd.DataFrame(injuries)
print(f"✅ Injuries loaded — {len(injuries_df)} players")

# ── 6. Build + save HTML ──────────────────────────────────────
html_newsletter = build_newsletter_html(next_round, this_round, ladder, odds_df, injuries_df, bye_teams)

with open("NRL_newsletter.html", "w", encoding="utf-8") as f:
    f.write(html_newsletter)

print(f"\n🎉 Done! NRL_newsletter.html saved")
print(f"   Round {int(latest_round)} recap + Round {next_round} preview ready")
print(f"   → Push NRL_newsletter.html to GitHub and you're live!")



✅ Match data loaded — latest completed round: 8
✅ Ladder built — 17 teams
✅ Odds loaded — 13 matches | Requests remaining: 495
✅ Injuries loaded — 75 players

🎉 Done! NRL_newsletter.html saved
   Round 8 recap + Round 9 preview ready
   → Push NRL_newsletter.html to GitHub and you're live!
